# Per-Feature IC (Test Set) — 1D vs 3D

This notebook computes **per-feature Rank IC / ICIR** (cross-sectional Spearman) against **forward returns** on the **test period**.

Use this *only as an interpretability appendix* (in case a professor asks "which features drive IC").

Notes:
- Per-feature IC is **model-agnostic** (depends on feature values + labels).
- Your reported `rank_ic` in `*_metrics.csv` is **model-score IC** (end-to-end).


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from src.data_loader import load_universe_prices, load_benchmark, load_macro, load_fundamentals
from src.features import build_feature_frame, feature_group_map
from src.labels import add_forward_labels

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)

ROOT = Path(r'C:/Users/user/Downloads/Moltbot/HKU-FYP/fyp_finance_ml_v2')
OUT = ROOT / 'outputs'

TAG = '084'
MODE = 'live'
metrics_path = OUT / 'metrics' / f'{TAG}_{MODE}_metrics.csv'
assert metrics_path.exists(), metrics_path
metrics = pd.read_csv(metrics_path)
row0 = metrics.iloc[0]
TEST_START = pd.to_datetime(row0['test_start'])
TEST_END = pd.to_datetime(row0['test_end'])
HORIZONS = sorted(metrics['horizon_days'].unique().tolist())
print('Test window:', TEST_START.date(), 'to', TEST_END.date())
print('Horizons:', HORIZONS)

## Load data + build features
This uses the same loaders/feature engineering as the pipeline.

In [ ]:
prices = load_universe_prices(mode=MODE)
bench = load_benchmark(mode=MODE)
macro = load_macro(mode=MODE)
fund = load_fundamentals(mode=MODE)

df = build_feature_frame(prices, bench, macro, fund)
df = add_forward_labels(df, horizons=HORIZONS)

df['date'] = pd.to_datetime(df['date'])
df = df[(df['date'] >= TEST_START) & (df['date'] <= TEST_END)].copy()
print('rows in test window:', len(df), 'tickers:', df['ticker'].nunique())
df[['date','ticker','close']].head()

## Per-feature Rank IC computation
For each day, compute Spearman correlation across tickers between a feature and the forward return. Then average across days; ICIR = mean / std.

In [ ]:
# Build a reverse map: feature -> group
gmap = feature_group_map()
feat_to_group = {}
for g, cols in gmap.items():
    for c in cols:
        feat_to_group[c] = g

feature_cols = [c for c in feat_to_group.keys() if c in df.columns]
print('feature cols available:', len(feature_cols))

def per_feature_ic(work: pd.DataFrame, feature: str, ret_col: str) -> pd.Series:
    # daily cross-sectional Spearman IC
    vals = []
    for dt, g in work.groupby('date'):
        x = g[feature]
        y = g[ret_col]
        tmp = pd.concat([x, y], axis=1).dropna()
        if len(tmp) < 10:  # require some breadth
            continue
        if tmp[feature].nunique() <= 1 or tmp[ret_col].nunique() <= 1:
            continue
        vals.append(tmp[feature].corr(tmp[ret_col], method='spearman'))
    return pd.Series(vals, dtype=float)

rows = []
for h in HORIZONS:
    ret_col = f'fwd_ret_{int(h)}d'
    for feat in feature_cols:
        s = per_feature_ic(df, feat, ret_col)
        ic_mean = float(s.mean()) if len(s) else np.nan
        ic_std = float(s.std(ddof=0)) if len(s) else np.nan
        icir = (ic_mean / ic_std) if (ic_std is not None and ic_std != 0 and not np.isnan(ic_std)) else np.nan
        rows.append({
            'horizon_days': int(h),
            'feature_group': feat_to_group.get(feat, 'unknown'),
            'feature': feat,
            'ic_mean': ic_mean,
            'ic_std': ic_std,
            'icir': icir,
            'n_days_used': int(len(s)),
        })

feat_ic = pd.DataFrame(rows)
feat_ic.head()

## Top features by IC / ICIR (1D and 3D)
These tables are screenshot-ready.

In [ ]:
def show_top(h, by='ic_mean', n=15):
    sub = feat_ic[feat_ic.horizon_days==h].copy()
    sub = sub.dropna(subset=[by])
    return sub.sort_values(by, ascending=False).head(n)[['feature_group','feature','ic_mean','icir','n_days_used']]

for h in HORIZONS:
    print('
=== Horizon', int(h), 'D | Top by IC mean ===')
    display(show_top(int(h), by='ic_mean', n=15))
    print('
=== Horizon', int(h), 'D | Top by ICIR ===')
    display(show_top(int(h), by='icir', n=15))

## Feature-group summary
Average IC within each feature group, for 1D and 3D.

In [ ]:
group_summary = (feat_ic.groupby(['horizon_days','feature_group'])
                 .agg(ic_mean_avg=('ic_mean','mean'), icir_avg=('icir','mean'), n_features=('feature','count'))
                 .reset_index()
                 .sort_values(['horizon_days','ic_mean_avg'], ascending=[True, False]))
group_summary

## Optional: export CSV (for backup / appendix)
If you want to save the per-feature IC table for later, run this cell.

In [ ]:
export_path = OUT / 'tables' / f'{TAG}_{MODE}_per_feature_ic_test_1d3d.csv'
feat_ic.to_csv(export_path, index=False)
print('Saved:', export_path)